# 03 — Silver (typed, conformed)

Shreds each bronze VARIANT into a typed `silver_<entity>` table, driven by the `(path, alias, type)` maps
in `synergy_schemas.SILVER_COLUMNS`. **To add columns/entities:** edit that file (paste the full projection
from the `mlb_pipelines` accelerator's `src/synergy/<entity>/endpoint.yml`).

> Uses `CREATE OR REPLACE TABLE` (full refresh) — works through Databricks Connect, where `MERGE` isn't
> supported. Inside a workspace cluster you can switch to `MERGE` for incremental loads.

**The conformance hook:** `teams.external_id_mlbam` is the MLBAM team id — the same key the broader MLB
warehouse joins on (`statsapi.silver.teams`, `gold.dim_team`). That's how this Synergy data ties into
everything else for the customer.

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from synergy_schemas import SILVER_COLUMNS, PRIMARY_KEYS

def build_silver(entity: str):
    cols = SILVER_COLUMNS[entity]
    # VARIANT navigation: `data:a:b::type`. Nested objects use `:` (e.g. data:homeTeam:division:id).
    # Clean data casts directly; if a messier entity has junk values that hard-fail a cast, swap
    # `data:{path}::{typ}` for `try_cast(data:{path}::string AS {typ})` (nulls bad values instead).
    select = ",\n        ".join(f"data:{path}::{typ} AS {alias}" for path, alias, typ in cols)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {S}.silver_{entity} AS
        SELECT
        {select},
        _ingestion_timestamp
        FROM {B}.bronze_{entity}
    """)
    # dedup to the natural key (latest ingest wins) so re-pulls converge
    pk = PRIMARY_KEYS[entity]
    on = " AND ".join(f"a.{k} <=> b.{k}" for k in pk)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {S}.silver_{entity} AS
        SELECT * EXCEPT(_rn) FROM (
          SELECT *, row_number() OVER (PARTITION BY {','.join(pk)} ORDER BY _ingestion_timestamp DESC) _rn
          FROM {S}.silver_{entity}
        ) WHERE _rn = 1
    """)
    print(f"silver_{entity}: {spark.table(f'{S}.silver_{entity}').count():,} rows")

for e in ["teams", "games"]:
    build_silver(e)

In [ ]:
# sample + the cross-source key
spark.sql(f"SELECT id, name, external_id_mlbam, league_name, division_name FROM {S}.silver_teams LIMIT 5").show(truncate=False)
spark.sql(f"SELECT id, season, game_date, home_team_name, away_team_name FROM {S}.silver_games LIMIT 5").show(truncate=False)